# Polarity

Solution Author: Cowille

In [1]:
import random, os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import BertTokenizerFast, BertForMaskedLM
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [2]:
seed = 2026
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
BASE = '/kaggle/input/competitions/synonym-antonym-discriminator-aicc-round-1'

train = pd.read_csv(f'{BASE}/train.csv')
test = pd.read_csv(f'{BASE}/test.csv')

print(f"train {train.shape}, test {test.shape}, label counts {train['label'].value_counts().to_dict()}")
train.head()

train (50, 3), test (686, 3), label counts {0: 25, 1: 25}


,w1,w2,label
0,cash,money,0
1,grab,take,0
2,irrational,rational,1
3,defend,guard,0
4,instructor,teacher,0


In [5]:
model_dir = f'{BASE}/bert-large-uncased'

tokenizer = BertTokenizerFast.from_pretrained(model_dir)
mlm = BertForMaskedLM.from_pretrained(model_dir).eval().to(device)
mask_id = tokenizer.mask_token_id

In [6]:
ant_templates = [
    '{w} but not [MASK] .',
    'the opposite of {w} is [MASK] .',
    '[MASK] is the opposite of {w} .',
    'either {w} or [MASK] .',
    'whether {w} or [MASK] .',
    'from {w} to [MASK] .',
    'neither {w} nor [MASK] .',
    'not {w} but [MASK] .',
    '{w} , not [MASK] .',
    '{w} versus [MASK] .',
    '{w} rather than [MASK] .',
    '{w} as opposed to [MASK] .',
    '{w} and [MASK] alike .',
    'the difference between {w} and [MASK] .',
    'the contrast between {w} and [MASK] .',
    'between {w} and [MASK] .',
    'more {w} than [MASK] .',
    'the {w} and the [MASK] .',
]
syn_templates = [
    '{w} means [MASK] .',
    'a synonym for {w} is [MASK] .',
    'another word for {w} is [MASK] .',
    '{w} , i . e . [MASK] .',
    '{w} , also called [MASK] .',
    '{w} , or [MASK] ,',
    '{w} that is [MASK] .',
    '{w} is the same as [MASK] .',
]

templates = ant_templates + syn_templates

In [7]:
prefix_rules = [('', 'ab'), ('', 'anti'), ('', 'dis'), ('', 'im'), ('', 'in'), ('', 'mal'), ('', 'mis'),
                ('', 'non'), ('', 'un'), ('l', 'ill'), ('r', 'ir'), ('im', 'ex'), ('in', 'ex'),
                ('up', 'down'), ('over', 'under')]
suffix_rules = [('less', 'ful')]

def is_affix_antonym(a, b):
    a, b = a.lower(), b.lower()
    for pa, pb in prefix_rules:
        for x, y in ((a, b), (b, a)):
            if x.startswith(pa) and y.startswith(pb) and x[len(pa):] == y[len(pb):] and len(x[len(pa):]) >= 3:
                return True
    for sa, sb in suffix_rules:
        for x, y in ((a, b), (b, a)):
            if x.endswith(sa) and y.endswith(sb) and x[:-len(sa)] == y[:-len(sb)] and len(x[:-len(sa)]) >= 3:
                return True
    return False

In [ ]:
@torch.inference_mode()
def mask_logprob(template, src_words, tgt_words):
    scores = []
    for i in range(0, len(src_words), 128):
        src, tgt = src_words[i:i + 128], tgt_words[i:i + 128]

        sentences = [template.format(w=w) for w in src]
        enc = tokenizer(sentences, return_tensors='pt', padding=True).to(device)
        logp = F.log_softmax(mlm(**enc).logits, dim=-1)

        mask_pos = (enc['input_ids'] == mask_id).float().argmax(1)
        tgt_ids = torch.tensor([tokenizer.convert_tokens_to_ids(w) for w in tgt], device=device)
        rows = torch.arange(len(src), device=device)

        scores.append(logp[rows, mask_pos, tgt_ids].float().cpu().numpy())
    return np.concatenate(scores)

def featurize(df):
    w1, w2 = list(df['w1']), list(df['w2'])
    return np.stack([(mask_logprob(t, w1, w2) + mask_logprob(t, w2, w1)) / 2 for t in templates], axis=1)

In [9]:
X_train, X_test = featurize(train), featurize(test)
y_train = train['label'].values

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"feature matrix {X_train.shape}")

feature matrix (50, 26)


In [10]:
clf = LogisticRegression(C=0.3, max_iter=3000).fit(X_train, y_train)

print(f"train accuracy {(clf.predict(X_train) == y_train).mean():.3f}")

train accuracy 0.980


In [11]:
pred = clf.predict(X_test)

override = np.array([is_affix_antonym(a, b) for a, b in zip(test['w1'], test['w2'])])
pred[override] = 1

print(f"{int(override.sum())} pairs set to antonym by the affix rule")

76 pairs set to antonym by the affix rule


In [12]:
submission = pd.DataFrame({'row_id': test['row_id'], 'label': pred.astype(int)})
submission.to_csv('submission.csv', index=False)

print(f"{int(submission['label'].sum())} antonyms predicted of {len(submission)}")
submission.head()

307 antonyms predicted of 686


,row_id,label
0,0,0
1,1,1
2,2,1
3,3,1
4,4,1
